# Pertemuan 3 — Brute Force, Exhaustive Search, dan Pencarian Ruang Keadaan
Bagian A: TSP brute force (O(n!)). Bagian B **[Modern]**: 8-puzzle dengan BFS, DFS, dan Iterative Deepening — O(bᵈ) dan trade-off waktu vs memori.

In [ ]:
import itertools, math, random, time
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
random.seed(1)

## A. TSP Brute Force

In [ ]:
def make_cities(n, seed=0):
    r = random.Random(seed)
    pts = [(r.random(), r.random()) for _ in range(n)]
    D = [[math.dist(p, q) for q in pts] for p in pts]
    return pts, D

def tsp_brute(D):
    n = len(D); best, best_p = float("inf"), None
    for perm in itertools.permutations(range(1, n)):      # kota 0 dikunci → (n-1)!
        c = D[0][perm[0]] + sum(D[perm[i]][perm[i+1]] for i in range(n-2)) + D[perm[-1]][0]
        if c < best: best, best_p = c, (0,) + perm
    return best, best_p

def tsp_nearest_neighbor(D):
    n = len(D); cur, seen, tot = 0, {0}, 0
    while len(seen) < n:
        nxt = min((j for j in range(n) if j not in seen), key=lambda j: D[cur][j])
        tot += D[cur][nxt]; seen.add(nxt); cur = nxt
    return tot + D[cur][0]

ns, ts = [], []
for n in range(4, 10):
    _, D = make_cities(n)
    t = time.perf_counter(); opt, _ = tsp_brute(D); dt = time.perf_counter() - t
    nn = tsp_nearest_neighbor(D); ns.append(n); ts.append(dt)
    print(f"n={n}  (n-1)!={math.factorial(n-1):>7}  waktu={dt:.3f}s  optimal={opt:.3f}  nearest-neighbor={nn:.3f} (+{100*(nn/opt-1):.1f}%)")
per_perm = ts[-1] / math.factorial(ns[-1] - 1)
for n in [12, 15, 20]:
    print(f"Ekstrapolasi n={n}: ≈ {per_perm*math.factorial(n-1)/3600/24/365:.2e} tahun")

## B. 8-Puzzle sebagai pencarian ruang keadaan

In [ ]:
GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)
def neighbors(s):
    i = s.index(0); r, c = divmod(i, 3)
    for dr, dc in ((-1,0),(1,0),(0,-1),(0,1)):
        nr, nc = r+dr, c+dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            j = nr*3+nc; l = list(s); l[i], l[j] = l[j], l[i]; yield tuple(l)

def scramble(k, seed=3):
    r = random.Random(seed); s, prev = GOAL, None
    for _ in range(k):
        opts = [x for x in neighbors(s) if x != prev]
        prev, s = s, r.choice(opts)
    return s

def bfs(start):
    q, seen, expanded, max_front = deque([(start, 0)]), {start}, 0, 1
    while q:
        s, d = q.popleft(); expanded += 1
        if s == GOAL: return d, expanded, max_front
        for nb in neighbors(s):
            if nb not in seen: seen.add(nb); q.append((nb, d+1))
        max_front = max(max_front, len(q))

def dfs_graph(start):                       # DFS dengan visited set (tidak optimal!)
    stack, seen, expanded, max_front = [(start, 0)], {start}, 0, 1
    while stack:
        s, d = stack.pop(); expanded += 1
        if s == GOAL: return d, expanded, max_front
        for nb in neighbors(s):
            if nb not in seen: seen.add(nb); stack.append((nb, d+1))
        max_front = max(max_front, len(stack))

def iddfs(start, max_depth=30):
    total = [0]
    def dls(s, d, limit, prev):
        total[0] += 1
        if s == GOAL: return True
        if d == limit: return False
        return any(dls(nb, d+1, limit, s) for nb in neighbors(s) if nb != prev)
    for limit in range(max_depth+1):
        if dls(start, 0, limit, None): return limit, total[0], limit   # memori O(d)

print(f"{'scramble':>8} | {'BFS depth/expanded/frontier':>30} | {'DFS depth/expanded':>20} | {'IDDFS depth/nodes':>18}")
for k in [6, 10, 14, 18]:
    s = scramble(k)
    b, d_, i = bfs(s), dfs_graph(s), iddfs(s)
    print(f"{k:>8} | {str(b):>30} | {str(d_[:2]):>20} | {str(i[:2]):>18}")

**Amati:** BFS optimal tetapi frontier (memori) tumbuh ~bᵈ; DFS hemat memori tetapi solusi bisa sangat panjang (tidak optimal); IDDFS optimal dengan memori O(d), dengan biaya node yang diulang (overhead konstan ≈ b/(b−1)).

## C. Latihan
1. Ganti heuristik: urutkan neighbor DFS secara acak, jalankan 20 kali — berapa variansi `expanded`?
2. Hitung faktor percabangan efektif b* = expanded^(1/d) untuk BFS.
3. Tambahkan **Uniform-Cost Search** dengan biaya langkah berbeda (mis. memindahkan angka bernilai v berbiaya v).
4. Tugas: tabel perbandingan metode untuk 5 konfigurasi awal (kedalaman 8–20) dan analisis O(bᵈ).

In [ ]:
def ucs(start):
    pass  # TODO